In [6]:
import pandas as pd

df = pd.read_csv('rba-dataset.csv', nrows=10000)
df.info()
df.head()







<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   index                     10000 non-null  int64  
 1   Login Timestamp           10000 non-null  str    
 2   User ID                   10000 non-null  int64  
 3   Round-Trip Time [ms]      392 non-null    float64
 4   IP Address                10000 non-null  str    
 5   Country                   10000 non-null  str    
 6   Region                    9996 non-null   str    
 7   City                      9998 non-null   str    
 8   ASN                       10000 non-null  int64  
 9   User Agent String         10000 non-null  str    
 10  Browser Name and Version  10000 non-null  str    
 11  OS Name and Version       10000 non-null  str    
 12  Device Type               10000 non-null  str    
 13  Login Successful          10000 non-null  bool   
 14  Is Attack IP      

,index,Login Timestamp,User ID,Round-Trip Time [ms],IP Address,Country,Region,City,ASN,User Agent String,Browser Name and Version,OS Name and Version,Device Type,Login Successful,Is Attack IP,Is Account Takeover
0,0,2020-02-03 12:43:30.772,-4324475583306591935,NaN,10.0.65.171,NO,-,-,29695,Mozilla/5.0 (iPhone; CPU iPhone OS 13_4 like ...,Firefox 20.0.0.1618,iOS 13.4,mobile,False,False,False
1,1,2020-02-03 12:43:43.549,-4324475583306591935,NaN,194.87.207.6,AU,-,-,60117,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile 46.0.2490,Android 4.1,mobile,False,False,False
2,2,2020-02-03 12:43:55.873,-3284137479262433373,NaN,81.167.144.58,NO,Vestland,Urangsvag,29695,Mozilla/5.0 (iPad; CPU OS 7_1 like Mac OS X) ...,Android 2.3.3.2672,iOS 7.1,mobile,True,False,False
3,3,2020-02-03 12:43:56.180,-4324475583306591935,NaN,170.39.78.152,US,-,-,393398,Mozilla/5.0 (Linux; Android 4.1; Galaxy Nexus...,Chrome Mobile WebView 85.0.4183,Android 4.1,mobile,False,False,False
4,4,2020-02-03 12:43:59.396,-4618854071942621186,NaN,10.0.0.47,US,Virginia,Ashburn,398986,Mozilla/5.0 (Linux; U; Android 2.2) Build/NMA...,Chrome Mobile WebView 85.0.4183,Android 2.2,mobile,False,True,False


In [7]:
df['Login Timestamp'] = pd.to_datetime(df['Login Timestamp'])
print("min date:", df['Login Timestamp'].min())
print("max date:", df['Login Timestamp'].max()) 

#even with 10k rows the timerange is just 2-3hrs span of same day

min date: 2020-02-03 12:43:30.772000
max date: 2020-02-03 14:42:21.689000


In [8]:
import duckdb
con = duckdb.connect()

path = '/home/igris/Documents/projects/MAJOR-PAIN-ATE-/data/raw/rba-dataset.csv'

df2 = con.execute(f"""
    SELECT * FROM (
        SELECT * FROM read_csv_auto('{path}')
        WHERE "Is Attack IP" = 'True'
        USING SAMPLE 5000 ROWS
    )
    UNION ALL
    SELECT * FROM (
        SELECT * FROM read_csv_auto('{path}')
        WHERE "Is Attack IP" = 'False'
        USING SAMPLE 45000 ROWS
    )
""").fetchdf()

df2['Login Timestamp'] = pd.to_datetime(df2['Login Timestamp'])

print("Rows:", len(df2))
print("Attack IP:", df2['Is Attack IP'].sum())
print("Attack %:", round(df2['Is Attack IP'].mean() * 100, 1))
print("Users:", df2['User ID'].nunique())
print("Min:", df2['Login Timestamp'].min())
print("Max:", df2['Login Timestamp'].max())
print("Avg events/user:", round(len(df2) / df2['User ID'].nunique(), 1))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 40985
Attack IP: 479
Attack %: 1.2
Users: 22588
Min: 2020-02-03 12:58:29.360000
Max: 2021-02-28 23:59:37.293000
Avg events/user: 1.8


In [9]:
df2 = df2.sort_values(['User ID', 'Login Timestamp'])

user_counts = df2['User ID'].value_counts()
active_users = user_counts[user_counts >= 3].index
df2 = df2[df2['User ID'].isin(active_users)].copy()
print(f"After filtering: {len(df2)} rows, {df2['User ID'].nunique()} users")

user_history = {}
features = []

for idx, row in df2.iterrows():
    uid = row['User ID']
    ts = row['Login Timestamp']
    device_key = f"{row['Device Type']}|{row['Browser Name and Version']}|{row['OS Name and Version']}"
    
    if uid not in user_history:
        user_history[uid] = {
            'countries': set(), 'devices': set(), 'last_60s': [],
            'fails_last_5min': 0, 'today_count': 0, 'last_date': None
        }
    
    hist = user_history[uid]
    hour = ts.hour
    is_night = 1 if hour < 6 or hour > 22 else 0
    is_weekend = 1 if ts.weekday() >= 5 else 0
    country_change = 1 if row['Country'] not in hist['countries'] else 0
    device_change = 1 if device_key not in hist['devices'] else 0
    
    failed_before = 0
    if row['Login Successful']:
        failed_before = hist['fails_last_5min']
    
    cutoff_60s = ts - pd.Timedelta(seconds=60)
    hist['last_60s'] = [t for t in hist['last_60s'] if t > cutoff_60s]
    rapid_rate = len(hist['last_60s'])
    
    if hist['last_date'] != ts.date():
        hist['today_count'] = 0
        hist['last_date'] = ts.date()
    hist['today_count'] += 1
    freq_today = hist['today_count']
    
    features.append({
        'hour': hour, 'is_night': is_night, 'is_weekend': is_weekend,
        'country_change': country_change, 'device_change': device_change,
        'failed_before_success': failed_before,
        'rapid_login_rate': rapid_rate, 'login_frequency_today': freq_today,
        'label': int(row['Is Attack IP'])
    })
    
    hist['countries'].add(row['Country'])
    hist['devices'].add(device_key)
    hist['last_60s'].append(ts)
    if not row['Login Successful']:
        hist['fails_last_5min'] += 1
    else:
        hist['fails_last_5min'] = 0

train_df = pd.DataFrame(features)
train_df.head(20)


After filtering: 18191 rows, 33 users


,hour,is_night,is_weekend,country_change,device_change,failed_before_success,rapid_login_rate,login_frequency_today,label
0,10,0,0,1,1,0,0,1,0
1,4,1,0,0,1,0,0,1,0
2,4,1,0,0,1,0,0,1,0
3,18,0,1,1,1,0,0,1,0
4,14,0,0,0,1,0,0,1,0
5,16,0,0,0,1,0,0,1,0
6,17,0,0,1,1,0,0,1,0
7,14,0,0,0,1,1,0,1,0
8,9,0,1,0,1,0,0,1,0
9,18,0,0,1,1,0,0,1,0


In [11]:
train_df.to_csv('training_data.csv', index=False)
print("Saved training_data.csv")
print("Shape:", train_df.shape)
print(train_df['label'].value_counts())

Saved training_data.csv
Shape: (18191, 9)
label
0    17943
1      248
Name: count, dtype: int64
